In [3]:
# ============================================================
# SetStop-KGQA Notebook 03 — Corrected Version
# Qwen2.5-0.5B QLoRA Marginal-Gain Controller
# Complete single-cell Google Colab code
# ============================================================

# IMPORTANT:
# Restart the Colab runtime before running this corrected cell.
# Runtime -> Restart session
# Hardware accelerator -> T4 GPU

# ============================================================
# 1. INSTALL DEPENDENCIES
# ============================================================

!pip -q install -U \
    transformers \
    datasets \
    accelerate \
    peft \
    bitsandbytes \
    scikit-learn \
    sentencepiece


# ============================================================
# 2. IMPORTS
# ============================================================

import gc
import hashlib
import inspect
import json
import math
import os
import random
import time

from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import torch
import transformers

from datasets import Dataset

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    brier_score_loss,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)

from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    set_seed,
)

from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
    prepare_model_for_kbit_training,
)

from google.colab import drive


# ============================================================
# 3. CLEAR PREVIOUS GPU OBJECTS
# ============================================================

for variable_name in [
    "trainer",
    "model",
    "tokenized_train",
    "tokenized_dev",
]:
    if variable_name in globals():
        del globals()[variable_name]

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


# ============================================================
# 4. MOUNT DRIVE
# ============================================================

drive.mount("/content/drive")


# ============================================================
# 5. CONFIGURATION
# ============================================================

SEED = 42

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

ROOT = Path(
    "/content/drive/MyDrive/SetStop_KGQA"
)

PROCESSED = ROOT / "data/processed"

TRAIN_STATE_PATH = (
    PROCESSED
    / "webqsp_train_states_v2_full.jsonl"
)

DEV_STATE_PATH = (
    PROCESSED
    / "webqsp_dev_states_v2_full.jsonl"
)

MODEL_ROOT = (
    ROOT
    / "models/setstop_qwen_gain_controller_v1"
)

OUTPUT_ROOT = (
    ROOT
    / "outputs/controller_training_v1"
)

CACHE_ROOT = (
    ROOT
    / "data/controller_cache_v1"
)

CHECKPOINT_ROOT = (
    OUTPUT_ROOT
    / "checkpoints"
)

for directory in [
    MODEL_ROOT,
    OUTPUT_ROOT,
    CACHE_ROOT,
    CHECKPOINT_ROOT,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

MAX_SEQUENCE_LENGTH = 256

# 15,000 examples from each of four classes.
MAX_TRAIN_PER_CLASS = 15_000

NUMBER_OF_LABELS = 4

LABEL_TO_ID = {
    "0": 0,
    "1": 1,
    "2": 2,
    "3_plus": 3,
}

ID_TO_LABEL = {
    0: "0",
    1: "1",
    2: "2",
    3: "3_plus",
}

TRAIN_CACHE = (
    CACHE_ROOT
    / "balanced_train_candidates_v1.jsonl"
)

# This cache has a new name because development data
# must retain every candidate for state-level calibration.
DEV_CACHE = (
    CACHE_ROOT
    / "all_dev_candidates_v2.jsonl"
)


# ============================================================
# 6. ENVIRONMENT CHECK
# ============================================================

print("=" * 90)
print("SETSTOP-KGQA GAIN CONTROLLER TRAINING")
print("=" * 90)

print("Transformers:", transformers.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

assert torch.cuda.is_available(), (
    "GPU is unavailable. Select a T4 GPU runtime."
)

GPU_NAME = torch.cuda.get_device_name(0)

GPU_MEMORY_GB = (
    torch.cuda.get_device_properties(0).total_memory
    / (1024 ** 3)
)

print("GPU:", GPU_NAME)
print(
    "GPU memory:",
    round(GPU_MEMORY_GB, 2),
    "GB"
)
print("Model:", MODEL_NAME)

assert TRAIN_STATE_PATH.exists(), (
    f"Missing training state file:\n"
    f"{TRAIN_STATE_PATH}"
)

assert DEV_STATE_PATH.exists(), (
    f"Missing development state file:\n"
    f"{DEV_STATE_PATH}"
)

print("Training states:", TRAIN_STATE_PATH)
print("Development states:", DEV_STATE_PATH)


# ============================================================
# 7. JSONL HELPERS
# ============================================================

def iter_jsonl(path):
    with open(
        path,
        encoding="utf-8"
    ) as file:

        for line_number, line in enumerate(
            file,
            start=1
        ):
            if not line.strip():
                continue

            try:
                yield json.loads(line)

            except Exception as error:
                raise ValueError(
                    f"Invalid JSON at "
                    f"{path}:{line_number}"
                ) from error


def write_jsonl(rows, path):
    with open(
        path,
        "w",
        encoding="utf-8"
    ) as file:

        for row in rows:
            file.write(
                json.dumps(
                    row,
                    ensure_ascii=False
                )
                + "\n"
            )


# ============================================================
# 8. CANDIDATE HELPERS
# ============================================================

def candidate_uid(state, candidate):
    content = (
        f"{state['qid']}|"
        f"{state.get('trajectory_id', 0)}|"
        f"{state['step']}|"
        f"{candidate['source']}|"
        f"{candidate['relation']}"
    )

    return hashlib.sha1(
        content.encode("utf-8")
    ).hexdigest()


def make_candidate_row(
    state,
    candidate
):
    gain_class = candidate["gain_class"]

    if gain_class not in LABEL_TO_ID:
        raise ValueError(
            f"Unexpected gain class: "
            f"{gain_class}"
        )

    return {
        "uid": candidate_uid(
            state,
            candidate
        ),
        "qid": state["qid"],
        "trajectory_id":
            state.get("trajectory_id", 0),
        "step": int(state["step"]),
        "text": candidate["model_input"],
        "label": LABEL_TO_ID[gain_class],
        "gain_class": gain_class,
        "stop_label":
            int(state["stop_label"]),
    }


# ============================================================
# 9. BALANCED TRAINING SAMPLING
# ============================================================

def sample_balanced_training_candidates(
    state_path,
    maximum_per_class
):
    rng = random.Random(SEED)

    reservoirs = {
        gain_class: []
        for gain_class in LABEL_TO_ID
    }

    observed_counts = Counter()

    state_count = 0
    candidate_count = 0

    for state in iter_jsonl(state_path):
        state_count += 1

        for candidate in state["candidates"]:
            gain_class = candidate["gain_class"]

            if gain_class not in LABEL_TO_ID:
                raise ValueError(
                    f"Unexpected class: "
                    f"{gain_class}"
                )

            observed_counts[gain_class] += 1
            candidate_count += 1

            row = make_candidate_row(
                state,
                candidate
            )

            class_seen = (
                observed_counts[gain_class]
            )

            reservoir = reservoirs[gain_class]

            if len(reservoir) < maximum_per_class:
                reservoir.append(row)

            else:
                replacement_index = rng.randrange(
                    class_seen
                )

                if replacement_index < maximum_per_class:
                    reservoir[
                        replacement_index
                    ] = row

        if state_count % 10_000 == 0:
            print(
                "Training states:",
                state_count,
                "| candidates:",
                candidate_count
            )

    sampled_rows = []

    for gain_class in [
        "0",
        "1",
        "2",
        "3_plus",
    ]:
        rng.shuffle(
            reservoirs[gain_class]
        )

        sampled_rows.extend(
            reservoirs[gain_class]
        )

    rng.shuffle(sampled_rows)

    summary = {
        "states_read": state_count,
        "candidates_read": candidate_count,
        "observed_class_counts":
            dict(observed_counts),
        "sampled_class_counts": dict(
            Counter(
                row["gain_class"]
                for row in sampled_rows
            )
        ),
        "sampled_total":
            len(sampled_rows),
    }

    return sampled_rows, summary


# ============================================================
# 10. LOAD EVERY DEVELOPMENT CANDIDATE
# ============================================================

def load_all_development_candidates(
    state_path
):
    rows = []
    observed_counts = Counter()

    state_count = 0
    candidate_count = 0

    for state in iter_jsonl(state_path):
        state_count += 1

        for candidate in state["candidates"]:
            row = make_candidate_row(
                state,
                candidate
            )

            rows.append(row)

            observed_counts[
                row["gain_class"]
            ] += 1

            candidate_count += 1

    summary = {
        "states_read": state_count,
        "candidates_read": candidate_count,
        "class_counts":
            dict(observed_counts),
        "total": len(rows),
    }

    return rows, summary


# ============================================================
# 11. PREPARE OR LOAD CACHED DATA
# ============================================================

if TRAIN_CACHE.exists():
    print(
        "\nLoading cached balanced "
        "training candidates..."
    )

    train_rows = list(
        iter_jsonl(TRAIN_CACHE)
    )

    train_summary = {
        "loaded_from_cache": True,
        "sampled_total": len(train_rows),
        "sampled_class_counts": dict(
            Counter(
                row["gain_class"]
                for row in train_rows
            )
        ),
    }

else:
    print(
        "\nSampling balanced "
        "training candidates..."
    )

    train_rows, train_summary = (
        sample_balanced_training_candidates(
            state_path=TRAIN_STATE_PATH,
            maximum_per_class=
                MAX_TRAIN_PER_CLASS,
        )
    )

    write_jsonl(
        train_rows,
        TRAIN_CACHE
    )


if DEV_CACHE.exists():
    print(
        "\nLoading cached complete "
        "development candidates..."
    )

    dev_rows = list(
        iter_jsonl(DEV_CACHE)
    )

    dev_summary = {
        "loaded_from_cache": True,
        "total": len(dev_rows),
        "class_counts": dict(
            Counter(
                row["gain_class"]
                for row in dev_rows
            )
        ),
    }

else:
    print(
        "\nLoading all development "
        "candidates..."
    )

    dev_rows, dev_summary = (
        load_all_development_candidates(
            DEV_STATE_PATH
        )
    )

    write_jsonl(
        dev_rows,
        DEV_CACHE
    )


print("\nTraining-data summary")
print(
    json.dumps(
        train_summary,
        indent=2
    )
)

print("\nDevelopment-data summary")
print(
    json.dumps(
        dev_summary,
        indent=2
    )
)

assert train_rows, (
    "No training candidates found"
)

assert dev_rows, (
    "No development candidates found"
)

train_question_ids = {
    row["qid"]
    for row in train_rows
}

dev_question_ids = {
    row["qid"]
    for row in dev_rows
}

assert train_question_ids.isdisjoint(
    dev_question_ids
), (
    "Question leakage detected between "
    "training and development"
)

print(
    "\nTrain questions:",
    len(train_question_ids)
)

print(
    "Development questions:",
    len(dev_question_ids)
)


# ============================================================
# 12. CREATE DATASETS
# ============================================================

train_dataset = Dataset.from_list([
    {
        "text": row["text"],
        "labels": int(row["label"]),
    }
    for row in train_rows
])

dev_dataset = Dataset.from_list([
    {
        "text": row["text"],
        "labels": int(row["label"]),
    }
    for row in dev_rows
])

print(
    "\nTrain examples:",
    len(train_dataset)
)

print(
    "Development examples:",
    len(dev_dataset)
)


# ============================================================
# 13. TOKENIZER
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = (
        tokenizer.eos_token
    )

tokenizer.padding_side = "right"


def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_SEQUENCE_LENGTH,
        padding=False,
    )


tokenized_train = train_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text"],
    desc="Tokenizing training data",
)

tokenized_dev = dev_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text"],
    desc="Tokenizing development data",
)

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    pad_to_multiple_of=8,
)


# ============================================================
# 14. LOAD QWEN IN 4-BIT
# ============================================================

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=
        torch.float16,
)

model = (
    AutoModelForSequenceClassification
    .from_pretrained(
        MODEL_NAME,
        num_labels=NUMBER_OF_LABELS,
        id2label=ID_TO_LABEL,
        label2id=LABEL_TO_ID,
        quantization_config=
            quantization_config,
        device_map={"": 0},
        dtype=torch.float16,
    )
)

model.config.pad_token_id = (
    tokenizer.pad_token_id
)

model.config.use_cache = False

model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
)

lora_configuration = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],
    modules_to_save=["score"],
)

model = get_peft_model(
    model,
    lora_configuration
)

model.print_trainable_parameters()


# ============================================================
# 15. METRIC HELPERS
# ============================================================

def extract_logits(predictions):
    if isinstance(predictions, tuple):
        return predictions[0]

    return predictions


def stable_softmax(logits):
    logits = np.asarray(
        logits,
        dtype=np.float64
    )

    shifted = (
        logits
        - np.max(
            logits,
            axis=1,
            keepdims=True
        )
    )

    exponentials = np.exp(shifted)

    return (
        exponentials
        / np.sum(
            exponentials,
            axis=1,
            keepdims=True
        )
    )


def compute_metrics(
    evaluation_prediction
):
    logits = extract_logits(
        evaluation_prediction.predictions
    )

    labels = (
        evaluation_prediction.label_ids
    )

    predictions = np.argmax(
        logits,
        axis=1
    )

    return {
        "accuracy": float(
            accuracy_score(
                labels,
                predictions
            )
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(
                labels,
                predictions
            )
        ),
        "macro_f1": float(
            f1_score(
                labels,
                predictions,
                average="macro",
                zero_division=0,
            )
        ),
        "weighted_f1": float(
            f1_score(
                labels,
                predictions,
                average="weighted",
                zero_division=0,
            )
        ),
    }


# ============================================================
# 16. VERSION-ADAPTIVE TRAINING ARGUMENTS
# ============================================================

training_argument_parameters = set(
    inspect.signature(
        TrainingArguments.__init__
    ).parameters
)

training_argument_values = {
    "output_dir":
        str(CHECKPOINT_ROOT),

    "num_train_epochs": 1,

    "per_device_train_batch_size": 4,
    "per_device_eval_batch_size": 8,

    "gradient_accumulation_steps": 8,

    "learning_rate": 2e-4,
    "weight_decay": 0.01,

    # Replaces the unsupported warmup_ratio.
    "warmup_steps": 100,

    "lr_scheduler_type": "cosine",

    "fp16": True,
    "bf16": False,

    "logging_strategy": "steps",
    "logging_steps": 25,

    "eval_steps": 500,

    "save_strategy": "steps",
    "save_steps": 500,
    "save_total_limit": 2,

    "load_best_model_at_end": True,
    "metric_for_best_model": "macro_f1",
    "greater_is_better": True,

    "gradient_checkpointing": True,
    "max_grad_norm": 1.0,

    "report_to": "none",

    "remove_unused_columns": True,

    "seed": SEED,
    "data_seed": SEED,

    "dataloader_num_workers": 2,
    "group_by_length": True,

    "optim": "paged_adamw_8bit",
}

# Transformers versions use one of these two names.
if (
    "eval_strategy"
    in training_argument_parameters
):
    training_argument_values[
        "eval_strategy"
    ] = "steps"

elif (
    "evaluation_strategy"
    in training_argument_parameters
):
    training_argument_values[
        "evaluation_strategy"
    ] = "steps"

else:
    raise RuntimeError(
        "This Transformers version exposes "
        "neither eval_strategy nor "
        "evaluation_strategy."
    )


# Keep only arguments supported by the installed release.
supported_training_arguments = {
    key: value
    for key, value
    in training_argument_values.items()
    if key in training_argument_parameters
}

ignored_training_arguments = sorted(
    set(training_argument_values)
    - set(supported_training_arguments)
)

print(
    "\nSupported TrainingArguments:",
    sorted(
        supported_training_arguments.keys()
    )
)

if ignored_training_arguments:
    print(
        "Ignored unsupported optional arguments:",
        ignored_training_arguments
    )


required_training_arguments = [
    "output_dir",
    "per_device_train_batch_size",
    "gradient_accumulation_steps",
    "learning_rate",
    "warmup_steps",
    "eval_steps",
    "save_steps",
]

for argument_name in required_training_arguments:
    assert (
        argument_name
        in supported_training_arguments
    ), (
        f"Required TrainingArguments field "
        f"is unavailable: {argument_name}"
    )


training_arguments = TrainingArguments(
    **supported_training_arguments
)


# ============================================================
# 17. VERSION-ADAPTIVE TRAINER
# ============================================================

trainer_parameters = set(
    inspect.signature(
        Trainer.__init__
    ).parameters
)

trainer_values = {
    "model": model,
    "args": training_arguments,
    "train_dataset": tokenized_train,
    "eval_dataset": tokenized_dev,
    "data_collator": data_collator,
    "compute_metrics": compute_metrics,
    "callbacks": [
        EarlyStoppingCallback(
            early_stopping_patience=2
        )
    ],
}

# New Transformers releases use processing_class.
if "processing_class" in trainer_parameters:
    trainer_values[
        "processing_class"
    ] = tokenizer

# Older releases use tokenizer.
elif "tokenizer" in trainer_parameters:
    trainer_values[
        "tokenizer"
    ] = tokenizer

else:
    print(
        "Trainer has no tokenizer or "
        "processing_class parameter."
    )


trainer = Trainer(
    **trainer_values
)


# ============================================================
# 18. TRAIN
# ============================================================

print("\nStarting QLoRA training...")
print(
    "Expected optimizer update steps:",
    math.ceil(
        len(train_dataset)
        / (
            training_arguments
            .per_device_train_batch_size
            * training_arguments
            .gradient_accumulation_steps
        )
    )
)

training_start = time.time()

training_result = trainer.train()

training_minutes = (
    time.time() - training_start
) / 60

print(
    "\nTraining time:",
    round(training_minutes, 2),
    "minutes"
)


# ============================================================
# 19. SAVE MODEL
# ============================================================

trainer.save_model(
    str(MODEL_ROOT)
)

tokenizer.save_pretrained(
    str(MODEL_ROOT)
)

training_metrics = {
    key: (
        float(value)
        if isinstance(
            value,
            (int, float, np.number)
        )
        else value
    )
    for key, value
    in training_result.metrics.items()
}

training_metrics[
    "training_minutes"
] = float(training_minutes)

training_metrics[
    "transformers_version"
] = transformers.__version__

training_metrics[
    "pytorch_version"
] = torch.__version__

training_metrics_path = (
    OUTPUT_ROOT
    / "training_metrics_v1.json"
)

training_metrics_path.write_text(
    json.dumps(
        training_metrics,
        indent=2
    ),
    encoding="utf-8",
)

print("Saved adapter:", MODEL_ROOT)


# ============================================================
# 20. DEVELOPMENT PREDICTIONS
# ============================================================

print(
    "\nRunning development predictions..."
)

development_output = trainer.predict(
    tokenized_dev
)

development_logits = extract_logits(
    development_output.predictions
)

development_labels = np.asarray(
    development_output.label_ids
)

development_probabilities = (
    stable_softmax(
        development_logits
    )
)

development_predictions = np.argmax(
    development_probabilities,
    axis=1
)


# ============================================================
# 21. CANDIDATE-LEVEL METRICS
# ============================================================

candidate_report = classification_report(
    development_labels,
    development_predictions,
    labels=[0, 1, 2, 3],
    target_names=[
        "gain_0",
        "gain_1",
        "gain_2",
        "gain_3_plus",
    ],
    output_dict=True,
    zero_division=0,
)

candidate_confusion_matrix = (
    confusion_matrix(
        development_labels,
        development_predictions,
        labels=[0, 1, 2, 3],
    ).tolist()
)

candidate_metrics = {
    "number_of_examples":
        int(len(development_labels)),

    "class_distribution": {
        ID_TO_LABEL[class_id]: int(
            np.sum(
                development_labels
                == class_id
            )
        )
        for class_id in range(4)
    },

    "accuracy": float(
        accuracy_score(
            development_labels,
            development_predictions
        )
    ),

    "balanced_accuracy": float(
        balanced_accuracy_score(
            development_labels,
            development_predictions
        )
    ),

    "macro_f1": float(
        f1_score(
            development_labels,
            development_predictions,
            average="macro",
            zero_division=0,
        )
    ),

    "weighted_f1": float(
        f1_score(
            development_labels,
            development_predictions,
            average="weighted",
            zero_division=0,
        )
    ),

    "classification_report":
        candidate_report,

    "confusion_matrix":
        candidate_confusion_matrix,
}

candidate_metrics_path = (
    OUTPUT_ROOT
    / "dev_candidate_metrics_v1.json"
)

candidate_metrics_path.write_text(
    json.dumps(
        candidate_metrics,
        indent=2
    ),
    encoding="utf-8",
)

print("\nCandidate-level metrics")
print(
    json.dumps(
        candidate_metrics,
        indent=2
    )
)


# ============================================================
# 22. STATE-LEVEL STOP CALIBRATION
# ============================================================

# Probability that a frontier has any positive gain.
positive_gain_probability = (
    1.0
    - development_probabilities[:, 0]
)

state_candidate_scores = defaultdict(
    list
)

state_true_labels = {}

for index, row in enumerate(dev_rows):
    state_key = (
        row["qid"],
        int(row["trajectory_id"]),
        int(row["step"]),
    )

    state_candidate_scores[
        state_key
    ].append(
        float(
            positive_gain_probability[index]
        )
    )

    state_true_labels[
        state_key
    ] = int(row["stop_label"])


maximum_state_gain_scores = []
true_stop_labels = []

for state_key, candidate_scores in (
    state_candidate_scores.items()
):
    maximum_state_gain_scores.append(
        max(candidate_scores)
    )

    true_stop_labels.append(
        state_true_labels[state_key]
    )


maximum_state_gain_scores = np.asarray(
    maximum_state_gain_scores,
    dtype=np.float64
)

true_stop_labels = np.asarray(
    true_stop_labels,
    dtype=np.int64
)

print(
    "\nDevelopment states used "
    "for calibration:",
    len(true_stop_labels)
)

print(
    "True STOP states:",
    int(np.sum(true_stop_labels))
)

print(
    "True CONTINUE states:",
    int(
        len(true_stop_labels)
        - np.sum(true_stop_labels)
    )
)


def evaluate_threshold(
    threshold
):
    predicted_stop = (
        maximum_state_gain_scores
        < threshold
    ).astype(np.int64)

    precision, recall, stop_f1, _ = (
        precision_recall_fscore_support(
            true_stop_labels,
            predicted_stop,
            average="binary",
            zero_division=0,
        )
    )

    premature_stop_rate = np.mean(
        (predicted_stop == 1)
        & (true_stop_labels == 0)
    )

    over_search_rate = np.mean(
        (predicted_stop == 0)
        & (true_stop_labels == 1)
    )

    return {
        "threshold": float(threshold),

        "stop_precision":
            float(precision),

        "stop_recall":
            float(recall),

        "stop_f1":
            float(stop_f1),

        "accuracy": float(
            accuracy_score(
                true_stop_labels,
                predicted_stop
            )
        ),

        "premature_stop_rate":
            float(premature_stop_rate),

        "over_search_rate":
            float(over_search_rate),
    }


threshold_results = [
    evaluate_threshold(threshold)
    for threshold in np.linspace(
        0.01,
        0.99,
        99
    )
]

best_threshold_result = max(
    threshold_results,
    key=lambda result: (
        result["stop_f1"],
        -result["premature_stop_rate"],
        result["accuracy"],
    )
)

selected_threshold = (
    best_threshold_result["threshold"]
)

predicted_stop_probability = (
    1.0
    - maximum_state_gain_scores
)

stop_brier_score = float(
    brier_score_loss(
        true_stop_labels,
        predicted_stop_probability
    )
)

calibration_output = {
    "selection_split": "development",

    "selection_rule": (
        "maximize STOP F1, then minimize "
        "premature stopping, then maximize "
        "accuracy"
    ),

    "selected_threshold":
        selected_threshold,

    "selected_threshold_metrics":
        best_threshold_result,

    "number_of_dev_states":
        int(len(true_stop_labels)),

    "number_of_true_stop_states":
        int(np.sum(true_stop_labels)),

    "number_of_true_continue_states":
        int(
            len(true_stop_labels)
            - np.sum(true_stop_labels)
        ),

    "stop_brier_score":
        stop_brier_score,

    "threshold_results":
        threshold_results,
}

threshold_path = (
    OUTPUT_ROOT
    / "dev_stop_threshold_v1.json"
)

threshold_path.write_text(
    json.dumps(
        calibration_output,
        indent=2
    ),
    encoding="utf-8",
)

print("\nStopping-threshold result")
print(
    json.dumps(
        {
            key: value
            for key, value
            in calibration_output.items()
            if key != "threshold_results"
        },
        indent=2
    )
)


# ============================================================
# 23. SAVE DEVELOPMENT PREDICTIONS
# ============================================================

prediction_path = (
    OUTPUT_ROOT
    / "dev_candidate_predictions_v1.jsonl"
)

with open(
    prediction_path,
    "w",
    encoding="utf-8"
) as output_file:

    for index, row in enumerate(dev_rows):
        prediction_row = {
            "uid": row["uid"],
            "qid": row["qid"],

            "trajectory_id":
                int(row["trajectory_id"]),

            "step":
                int(row["step"]),

            "stop_label":
                int(row["stop_label"]),

            "true_gain_label":
                int(development_labels[index]),

            "predicted_gain_label":
                int(
                    development_predictions[index]
                ),

            "probability_gain_0": float(
                development_probabilities[
                    index,
                    0
                ]
            ),

            "probability_gain_1": float(
                development_probabilities[
                    index,
                    1
                ]
            ),

            "probability_gain_2": float(
                development_probabilities[
                    index,
                    2
                ]
            ),

            "probability_gain_3_plus": float(
                development_probabilities[
                    index,
                    3
                ]
            ),

            "positive_gain_probability":
                float(
                    positive_gain_probability[
                        index
                    ]
                ),
        }

        output_file.write(
            json.dumps(prediction_row)
            + "\n"
        )


# ============================================================
# 24. FINAL SUMMARY
# ============================================================

run_summary = {
    "model": MODEL_NAME,
    "adapter_path": str(MODEL_ROOT),

    "transformers_version":
        transformers.__version__,

    "pytorch_version":
        torch.__version__,

    "gpu": GPU_NAME,
    "gpu_memory_gb":
        round(GPU_MEMORY_GB, 3),

    "train_examples":
        len(train_rows),

    "development_examples":
        len(dev_rows),

    "development_states":
        len(true_stop_labels),

    "maximum_sequence_length":
        MAX_SEQUENCE_LENGTH,

    "training_minutes":
        float(training_minutes),

    "candidate_accuracy":
        candidate_metrics["accuracy"],

    "candidate_balanced_accuracy":
        candidate_metrics[
            "balanced_accuracy"
        ],

    "candidate_macro_f1":
        candidate_metrics["macro_f1"],

    "selected_stop_threshold":
        selected_threshold,

    "selected_stop_f1":
        best_threshold_result[
            "stop_f1"
        ],

    "stop_precision":
        best_threshold_result[
            "stop_precision"
        ],

    "stop_recall":
        best_threshold_result[
            "stop_recall"
        ],

    "premature_stop_rate":
        best_threshold_result[
            "premature_stop_rate"
        ],

    "over_search_rate":
        best_threshold_result[
            "over_search_rate"
        ],

    "stop_brier_score":
        stop_brier_score,
}

summary_path = (
    OUTPUT_ROOT
    / "controller_training_summary_v1.json"
)

summary_path.write_text(
    json.dumps(
        run_summary,
        indent=2
    ),
    encoding="utf-8",
)

print("\n" + "=" * 90)
print("SETSTOP CONTROLLER TRAINING COMPLETED")
print("=" * 90)

print(
    json.dumps(
        run_summary,
        indent=2
    )
)

print("\nSaved adapter:")
print(MODEL_ROOT)

print("\nSaved outputs:")
print(training_metrics_path)
print(candidate_metrics_path)
print(threshold_path)
print(prediction_path)
print(summary_path)

print("=" * 90)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
SETSTOP-KGQA GAIN CONTROLLER TRAINING
Transformers: 5.17.0
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB
Model: Qwen/Qwen2.5-0.5B-Instruct
Training states: /content/drive/MyDrive/SetStop_KGQA/data/processed/webqsp_train_states_v2_full.jsonl
Development states: /content/drive/MyDrive/SetStop_KGQA/data/processed/webqsp_dev_states_v2_full.jsonl

Loading cached balanced training candidates...

Loading all development candidates...

Training-data summary
{
  "loaded_from_cache": true,
  "sampled_total": 60000,
  "sampled_class_counts": {
    "2": 15000,
    "0": 15000,
    "3_plus": 15000,
    "1": 15000
  }
}

Development-data summary
{
  "states_read": 969,
  "candidates_read": 15426,
  "class_counts": {
    "2": 3456,
    "0": 3189,
    "3_plus": 6892,
    "1": 1889
  },
  "total": 15426
}

Train questions: 1200
Development quest

Tokenizing training data:   0%|          | 0/60000 [00:00<?, ? examples/s]

Tokenizing development data:   0%|          | 0/15426 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-0.5B-Instruct
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


...

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss,Validation Loss


Step,Training Loss,Validation Loss,Accuracy,Balanced Accuracy,Macro F1,Weighted F1
500,4.624128,1.258968,0.592117,0.635616,0.586450,0.596346
1000,2.594906,1.064638,0.694542,0.687721,0.657810,0.687331
1500,1.919921,1.044823,0.722806,0.709317,0.689447,0.710267
1875,1.871997,1.034939,0.724037,0.715604,0.696530,0.715107



Training time: 126.86 minutes
Saved adapter: /content/drive/MyDrive/SetStop_KGQA/models/setstop_qwen_gain_controller_v1

Running development predictions...



Candidate-level metrics
{
  "number_of_examples": 15426,
  "class_distribution": {
    "0": 3189,
    "1": 1889,
    "2": 3456,
    "3_plus": 6892
  },
  "accuracy": 0.7240373395565928,
  "balanced_accuracy": 0.7156039115590327,
  "macro_f1": 0.6965298972691348,
  "weighted_f1": 0.7151074299442371,
  "classification_report": {
    "gain_0": {
      "precision": 0.769650655021834,
      "recall": 0.8842897460018815,
      "f1-score": 0.8229972274916095,
      "support": 3189.0
    },
    "gain_1": {
      "precision": 0.6142433234421365,
      "recall": 0.7670725251455797,
      "f1-score": 0.6822033898305084,
      "support": 1889.0
    },
    "gain_2": {
      "precision": 0.6085319949811794,
      "recall": 0.4210069444444444,
      "f1-score": 0.4976911236531555,
      "support": 3456.0
    },
    "gain_3_plus": {
      "precision": 0.7765259555048488,
      "recall": 0.7900464306442252,
      "f1-score": 0.7832278481012658,
      "support": 6892.0
    },
    "accuracy": 0.72403733